### Calculate Metrics (KGE, NSE, PBIAS)

For multiple experimental runs, you can process each input CSV and generate corresponding output CSVs with the Issue flag:

In [4]:
## Working function - 2
# -*- coding: utf-8 -*-
"""
Hydrology Model Evaluation Script
- Multiple station filters
- Consistent stations across plots (intersection across runs)
- Vectorized metrics
- YEAR+JDAY date filtering
- Improved eCDF/CDF plots with:
    * metric-aware scaling
    * outlier inset support
    * symmetric PBias scaling
    * NKGE fixed ranges
    * median/mean reference lines
- Spatial plots with basin overlay
"""

import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)
from pathlib import Path
from typing import List, Dict, Tuple, Optional, Set
from itertools import cycle
import re

import numpy as np
import pandas as pd
import geopandas as gpd

from mpl_toolkits.axes_grid1 import make_axes_locatable
import matplotlib.colors as colors
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from typing import List, Optional, Set

# =============================================================================
# METRIC DESCRIPTIONS
# =============================================================================
METRIC_DESCRIPTIONS: Dict[str, Dict[str, str]] = {
    'KGE': {'Name': 'Kling-Gupta Efficiency', 'Description': 'Combines correlation, bias, variability.',
            'Range': '(-∞,1]', 'Units': 'Dimensionless', 'Interpretation': 'Higher better'},

    'NKGE': {'Name': 'Normalized KGE', 'Description': 'Normalized form of KGE',
             'Range': '(0,1]', 'Units': 'Dimensionless',
             'Interpretation': 'Higher better'},

    'KGE2012': {'Name': 'KGE 2012',
                'Description': 'Modified KGE using coefficient of variation',
                'Range': '(-∞,1]', 'Units': 'Dimensionless',
                'Interpretation': 'Higher better'},

    'NKGE2012': {'Name': 'Normalized KGE 2012',
                 'Description': 'Normalized KGE 2012',
                 'Range': '(0,1]', 'Units': 'Dimensionless',
                 'Interpretation': 'Higher better'},

    'NSE': {'Name': 'Nash-Sutcliffe Efficiency',
            'Description': 'Variance explained',
            'Range': '(-∞,1]',
            'Units': 'Dimensionless',
            'Interpretation': 'Higher better'},

    'PBIAS': {'Name': 'Percent Bias',
              'Description': 'Mean bias',
              'Range': '(-∞,∞)',
              'Units': '%',
              'Interpretation': 'Closer to 0 better'},

    'RMSE': {'Name': 'Root Mean Square Error',
             'Description': 'Error magnitude',
             'Range': '[0,∞)',
             'Units': 'units',
             'Interpretation': 'Lower better'},

    'MAE': {'Name': 'Mean Absolute Error',
            'Description': 'Absolute error',
            'Range': '[0,∞)',
            'Units': 'units',
            'Interpretation': 'Lower better'},

    'R2': {'Name': 'Coefficient of Determination',
           'Description': 'Variance explained',
           'Range': '[0,1]',
           'Units': 'Dimensionless',
           'Interpretation': 'Higher better'},

    'MAPE': {'Name': 'Mean Absolute Percentage Error',
             'Description': 'Relative error',
             'Range': '[0,∞)',
             'Units': '%',
             'Interpretation': 'Lower better'},

    'VE': {'Name': 'Volume Error',
           'Description': 'Volumetric bias',
           'Range': '(-∞,∞)',
           'Units': '%',
           'Interpretation': 'Closer to 0 better'}
}

# =============================================================================
# METRIC CALCULATION
# =============================================================================
def compute_metrics_vectorized(sim: pd.Series, obs: pd.Series):

    df = pd.DataFrame({'sim': sim, 'obs': obs}).dropna()

    if len(df) < 2:
        return {m: np.nan for m in METRIC_DESCRIPTIONS}, "Insufficient data"

    sim, obs = df['sim'], df['obs']

    metrics, issue = {}, None

    mean_sim, mean_obs = sim.mean(), obs.mean()

    std_sim, std_obs = sim.std(ddof=1), obs.std(ddof=1)

    sum_obs = obs.sum()

    # ------------------------------------------------------------------
    # KGE + NKGE
    # ------------------------------------------------------------------
    if mean_obs == 0 or std_obs == 0 or std_sim == 0:

        metrics['KGE'] = np.nan
        metrics['NKGE'] = np.nan
        metrics['KGE2012'] = np.nan
        metrics['NKGE2012'] = np.nan

        issue = "Zero mean/std in KGE"

    else:

        r = np.corrcoef(sim, obs)[0, 1]

        if np.isnan(r):

            metrics['KGE'] = np.nan
            metrics['NKGE'] = np.nan
            metrics['KGE2012'] = np.nan
            metrics['NKGE2012'] = np.nan

            issue = "Invalid correlation"

        else:

            alpha = std_sim / std_obs

            beta = mean_sim / mean_obs

            gamma = (std_sim / mean_sim) / (std_obs / mean_obs)

            kge = 1 - np.sqrt(
                (r - 1) ** 2 +
                (alpha - 1) ** 2 +
                (beta - 1) ** 2
            )

            metrics['KGE'] = kge

            metrics['NKGE'] = 1 / (2 - kge)

            kge2012 = 1 - np.sqrt(
                (r - 1) ** 2 +
                (gamma - 1) ** 2 +
                (beta - 1) ** 2
            )

            metrics['KGE2012'] = kge2012

            metrics['NKGE2012'] = 1 / (2 - kge2012)

    # ------------------------------------------------------------------
    # NSE
    # ------------------------------------------------------------------
    if std_obs == 0:

        metrics['NSE'] = np.nan

        issue = issue or "Zero std in NSE"

    else:

        metrics['NSE'] = 1 - (
            ((obs - sim) ** 2).sum() /
            ((obs - mean_obs) ** 2).sum()
        )

    # ------------------------------------------------------------------
    # PBIAS + VE
    # ------------------------------------------------------------------
    if sum_obs == 0:

        metrics['PBIAS'] = np.nan
        metrics['VE'] = np.nan

        issue = issue or "Zero sum obs"

    else:

        bias = (sim.sum() - sum_obs) * 100 / sum_obs

        metrics['PBIAS'] = bias
        metrics['VE'] = bias

    metrics['RMSE'] = np.sqrt(((obs - sim) ** 2).mean())

    metrics['MAE'] = (obs - sim).abs().mean()

    # ------------------------------------------------------------------
    # R2
    # ------------------------------------------------------------------
    if std_obs == 0 or std_sim == 0:

        metrics['R2'] = np.nan

        issue = issue or "Zero std in R2"

    else:

        r = np.corrcoef(sim, obs)[0, 1]

        metrics['R2'] = r ** 2 if not np.isnan(r) else np.nan

    # ------------------------------------------------------------------
    # MAPE
    # ------------------------------------------------------------------
    nonzero = obs != 0

    if nonzero.sum() < 2:

        metrics['MAPE'] = np.nan

        issue = issue or "Insufficient non-zero obs"

    else:

        metrics['MAPE'] = (
            (
                (obs[nonzero] - sim[nonzero]).abs() /
                obs[nonzero].abs()
            ).mean() * 100
        )

    return metrics, issue


# =============================================================================
# PROCESSING (ADDED FROM FUNCTION 2 ✅)
# =============================================================================
def compute_efficiency_custom(
    df: pd.DataFrame,
    prefix_obs='QOMEAS_',
    prefix_simu='QOSIM_'
) -> pd.DataFrame:

    results = []

    for col_obs in [c for c in df.columns if c.startswith(prefix_obs)]:

        sid = col_obs[len(prefix_obs):].strip()

        col_sim = f"{prefix_simu}{sid}"

        if col_sim not in df.columns:
            results.append({
                'StationID': sid,
                **{m: np.nan for m in METRIC_DESCRIPTIONS},
                'Issue': True,
                'IssueMessage': 'Missing simulated column'
            })
            continue

        metrics, issue = compute_metrics_vectorized(
            df[col_sim],
            df[col_obs]
        )

        row = {
            'StationID': str(sid),
            **metrics,
            'Issue': issue is not None,
            'IssueMessage': issue or ''
        }

        results.append(row)

    return (
        pd.DataFrame(results)
        .sort_values('StationID')
        .reset_index(drop=True)
    )


def save_metric_descriptions(outdir: Path, filename='metrics_description.txt'):

    outdir = Path(outdir)
    outdir.mkdir(parents=True, exist_ok=True)

    p = outdir / filename

    with p.open('w', encoding='utf-8') as f:

        f.write("Hydrology Metrics Description\n")
        f.write("=" * 30 + "\n\n")

        for m, info in METRIC_DESCRIPTIONS.items():

            f.write(f"{info['Name']} ({m})\n")

            for k in ['Description', 'Range', 'Units', 'Interpretation']:
                f.write(f"{k}: {info[k]}\n")

            f.write("-" * 30 + "\n\n")


def process_flow_csv(
    input_csv: Path,
    output_csv: Path,
    prefix_obs='QOMEAS_',
    prefix_simu='QOSIM_',
    skip_days=0,
    missing_value=None,
    date_column="Date",
    start_date: Optional[str] = None,
    end_date: Optional[str] = None
):

    input_csv = Path(input_csv)
    output_csv = Path(output_csv)

    output_csv.parent.mkdir(parents=True, exist_ok=True)

    if not input_csv.exists():
        raise FileNotFoundError(input_csv)

    df = pd.read_csv(input_csv, sep=r',\s*', engine='python')
    # df = pd.read_csv(input_csv)

    # Replace missing values
    if missing_value is not None:
        df = df.replace(missing_value, np.nan)

    # ------------------------------------------------------------
    # DATE FILTERING
    # ------------------------------------------------------------
    if "YEAR" in df.columns and "JDAY" in df.columns:

        try:
            df["Date"] = (
                pd.to_datetime(df["YEAR"].astype(str), format="%Y") +
                pd.to_timedelta(df["JDAY"] - 1, unit="D")
            )

            if start_date:
                df = df[df["Date"] >= pd.to_datetime(start_date)]

            if end_date:
                df = df[df["Date"] <= pd.to_datetime(end_date)]

            df = df.reset_index(drop=True)

        except Exception as e:
            print(f"Warning: YEAR/JDAY parsing failed: {e}")

    elif date_column in df.columns:

        df[date_column] = pd.to_datetime(df[date_column], errors='coerce')

        if start_date:
            df = df[df[date_column] >= pd.to_datetime(start_date)]

        if end_date:
            df = df[df[date_column] <= pd.to_datetime(end_date)]

        df = df.reset_index(drop=True)

    else:
        print("Warning: No date column found; skipping date filtering.")

    # ------------------------------------------------------------
    # SPIN-UP REMOVAL
    # ------------------------------------------------------------
    if skip_days > 0:

        if skip_days >= len(df):
            raise ValueError("skip_days exceeds number of rows")

        df = df.iloc[skip_days:].reset_index(drop=True)

    # ------------------------------------------------------------
    # COMPUTE METRICS ✅
    # ------------------------------------------------------------
    out = compute_efficiency_custom(df, prefix_obs, prefix_simu)

    out['StationID'] = out['StationID'].astype(str).str.strip()

    out.to_csv(output_csv, index=False)

    # Save descriptions
    save_metric_descriptions(output_csv.parent)

    # Report issues
    n_issues = out['Issue'].sum()

    if n_issues > 0:
        print(f"⚠ {n_issues} stations had issues in {output_csv.name}")


def process_all_runs(
    base_path: Path,
    mesh_versions: List[str],
    gru_types: List[str],
    skip_days=10,
    missing_value=None,
    date_column="Date",
    start_date: Optional[str] = None,
    end_date: Optional[str] = None
) -> List[Path]:

    base_path = Path(base_path)

    outputs: List[Path] = []

    for mesh in mesh_versions:
        for gru in gru_types:

            folder = base_path / mesh / gru

            if not folder.exists():
                continue

            for sub in folder.iterdir():

                if not sub.is_dir():
                    continue

                input_csv = sub / "MESH_output_streamflow.csv"

                output_csv = sub / f"metrics_{mesh}_{gru}_{sub.name}.csv"

                if not input_csv.exists():
                    continue

                print(f"Processing: {input_csv.name}")

                process_flow_csv(
                    input_csv,
                    output_csv,
                    skip_days=skip_days,
                    missing_value=missing_value,
                    date_column=date_column,
                    start_date=start_date,
                    end_date=end_date
                )

                outputs.append(output_csv)

    return outputs

# =============================================================================
# STATION FILTERING
# =============================================================================
FilterCondition = Tuple[str, str]

def apply_station_filters(gdf: gpd.GeoDataFrame, filters: Optional[List[FilterCondition]]):
    if not filters:
        return gdf.copy()
    out = gdf.copy()
    for col, cond in filters:
        if col not in out.columns:
            raise ValueError(f"{col} missing")
        col_esc = f"`{col}`" if not re.match(r'^[A-Za-z_][A-Za-z0-9_]*$', col) else col
        try:
            before = len(out)
            out = out.query(f"{col_esc} {cond}")
            print(f"Filter {col} {cond}: {before}→{len(out)}")
        except Exception as e:
            raise ValueError(f"Invalid filter {col} {cond}: {e}")
    return out

def load_stations_with_filters(stations_file: Path, station_id_col: str, filters: Optional[List[FilterCondition]] = None) -> pd.DataFrame:
    stations_file = Path(stations_file)
    gdf = gpd.read_file(stations_file)
    if gdf.crs and gdf.crs.to_string() != "EPSG:4326":
        gdf = gdf.to_crs(4326)
    gdf = apply_station_filters(gdf, filters or [])
    return pd.DataFrame({
        'StationID': gdf[station_id_col].astype(str),
        'Longitude': gdf.geometry.x,
        'Latitude': gdf.geometry.y
    })

# =============================================================================
# METRICS CACHE
# =============================================================================
class MetricsCache:
    def __init__(self, csv_files: List[Path]):
        self.csv_files: List[Path] = [Path(p) for p in csv_files]
        self.dfs: List[pd.DataFrame] = []
        self.run_names: List[str] = []
        self._load_all()

    def _load_all(self):
        print(f"Loading {len(self.csv_files)} metric files...")
        for csv in self.csv_files:
            if not Path(csv).exists():
                print(f"Missing: {csv}")
                continue
            df = pd.read_csv(csv)
            if 'StationID' not in df.columns:
                print(f"Skipping invalid: {Path(csv).name}")
                continue
            df['StationID'] = df['StationID'].astype(str)
            name = Path(csv).stem.replace('metrics_', '').replace('_', ' ').title()
            self.run_names.append(name)
            self.dfs.append(df)
        print(f"Loaded {len(self.dfs)} valid runs.")

    def get_metric_data(self, metric: str, allowed_stations: Optional[Set[str]] = None):
        if metric not in METRIC_DESCRIPTIONS:
            raise ValueError(f"Invalid metric: {metric}")
        if not self.dfs:
            raise ValueError("No metric data loaded.")

        allowed_set: Optional[Set[str]] = set(allowed_stations) if allowed_stations is not None else None

        valid_sets: List[Set[str]] = []
        filtered_dfs: List[pd.DataFrame] = []

        for csv, df in zip(self.csv_files, self.dfs):
            if metric not in df.columns:
                raise ValueError(f"Metric '{metric}' missing in {Path(csv).name}")
            mask = df['StationID'].isin(allowed_set) if allowed_set is not None else slice(None)
            f = df.loc[mask].copy()
            f['StationID'] = f['StationID'].astype(str)
            valid = set(f[f[metric].notna()]['StationID'])
            valid_sets.append(valid)
            filtered_dfs.append(f)

        common: Set[str] = set.intersection(*valid_sets) if valid_sets else set()
        if allowed_set is not None:
            common &= allowed_set

        all_stations = set().union(*[set(df['StationID'].astype(str)) for df in self.dfs]) if self.dfs else set()
        excluded = all_stations - common
        if excluded:
            excl_file = Path(self.csv_files[0]).parent / f"excluded_stations_{metric.lower()}.txt"
            excl_file.write_text("\n".join(sorted(excluded)), encoding='utf-8')
            print(f"Excluded {len(excluded)} stations → {excl_file.name}")

        # Also persist the included (common) stations for transparency
        if self.csv_files:
            inc_file = Path(self.csv_files[0]).parent / f"common_stations_{metric.lower()}.txt"
            inc_file.write_text("\n".join(sorted(common)), encoding='utf-8')
            print(f"Common {len(common)} stations → {inc_file.name}")

        return filtered_dfs, common, self.run_names

    def common_stations(self, metric: str, allowed_stations: Optional[Set[str]] = None) -> Set[str]:
        """Convenience: intersection across runs for a metric, intersected with allowed stations if given."""
        _, common, _ = self.get_metric_data(metric, allowed_stations=allowed_stations)
        return common

# =============================================================================
# PLOTTING UTILITIES
# =============================================================================
def _metric_unit(metric: str) -> str:
    if metric in ["PBIAS", "MAPE", "VE"]:
        return "(%)"
    if metric in ["RMSE", "MAE"]:
        return "(units)"
    return ""


# =============================================================================
# OUTLIER INSET (ROBUST + FIXED)
# =============================================================================
def add_outlier_inset(ax, outlier_series):

    valid = [s for s in outlier_series if len(s["x"]) > 0]

    if not valid:
        return
                         #[left, bottom, width, height]   
    inset = ax.inset_axes([0.06, 0.60, 0.32, 0.32])

    xmin = min(np.min(s["x"]) for s in valid)
    xmax = max(np.max(s["x"]) for s in valid)
    ymax = max(np.max(s["y"]) for s in valid)

    for s in valid:
        inset.scatter(
            s["x"], s["y"],
            s=18,
            color=s["color"],
            edgecolors="k",
            linewidths=0.4,
            alpha=0.85,
            zorder=3,
        )

    inset.set_xlim(xmin, xmax)
    inset.set_ylim(-0.02, ymax + 0.05)

    inset.set_title(
        f"{sum(len(s['x']) for s in valid)} outlier(s)",
        fontsize=7,
        pad=2,
    )

    inset.tick_params(labelsize=6)
    inset.grid(True, alpha=0.3)

#    if len(valid) > 1:
#        inset.legend(
#            fontsize=7,
#            loc="upper left",
#            frameon=True,
#            framealpha=0.8,
#            edgecolor="black"
#        )

# =============================================================================
# CDF / eCDF PLOT (FINAL FIXED VERSION)
# =============================================================================
def plot_cumulative_distribution(
    cache,
    metric='KGE',
    output_file='cdf.png',
    common_stations: Optional[Set[str]] = None,
    legend_title: str = "Runs",
    legend_labels: Optional[List[str]] = None,
    line_colors: Optional[List[str]] = None,   # ✅ NEW
    line_width: float = 2.5                   # ✅ NEW
):

    output_file = Path(output_file)

    dfs, common, names = cache.get_metric_data(
        metric,
        allowed_stations=common_stations
    )

    if legend_labels and len(legend_labels) == len(names):
        names = legend_labels

    if not common:
        print("No common stations.")
        return

    metric_upper = metric.upper()

    cfg = {
        "KGE": {"inset": True, "clip": (-1, 1)},
        "KGE2012": {"inset": True, "clip": (-1, 1)},
        "NSE": {"inset": True, "clip": (-1, 1)},
        "NKGE": {"inset": False, "clip": (0, 1)},
        "NKGE2012": {"inset": False, "clip": (0, 1)},
        "R2": {"inset": False, "clip": (0, 1)},
        "PBIAS": {"inset": True, "clip": None},
        "VE": {"inset": True, "clip": None},
    }.get(metric_upper, {"inset": False, "clip": None})

    fig, ax = plt.subplots(figsize=(10 + len(dfs)//3, 6))

    # ✅ Color control
    if line_colors and len(line_colors) >= len(dfs):
        colors = line_colors
    else:
        colors = plt.cm.tab10(np.linspace(0, 1, max(len(dfs), 10)))

    outlier_series = []

    # ------------------------------------------------------------
    # MAIN LOOP (SOLID + PER-RUN STATS)
    # ------------------------------------------------------------
    for i, (df, name) in enumerate(zip(dfs, names)):

        data = df[df['StationID'].isin(common)][metric].dropna()
        if data.empty:
            continue

        vals = np.sort(data.values)
        ecdf_y = np.arange(1, len(vals) + 1) / len(vals)

        color = colors[i % len(colors)]

        # ✅ ECDF (SOLID)
        ax.step(
            vals,
            ecdf_y,
            where="post",
            label=name,
            color=color,
            linestyle='-',
            linewidth=line_width,
        )

        # ✅ PER-RUN MEDIAN + MEAN
        metric_upper = metric.upper()

        if metric_upper in ["NKGE", "NKGE2012"]:

            # ✅ Mean ONLY
            mean = np.mean(vals)

            ax.axvline(
                mean,
                color=color,
                linestyle=":",
                linewidth=1.6,
                alpha=0.95,
                zorder=3,
                label=None
            )

        else:

            # ✅ Median ONLY
            median = np.median(vals)

            ax.axvline(
                median,
                color=color,
                linestyle="--",
                linewidth=1.6,
                alpha=0.95,
                zorder=3,
                label=None
            )

        # ✅ OUTLIERS
        if cfg["inset"]:
            low, high = np.percentile(vals, [5, 95])
            mask = (vals <= low) | (vals >= high)

            outlier_series.append({
                "x": vals[mask],
                "y": ecdf_y[mask],
                "color": color,
            })

    # ------------------------------------------------------------
    # AXIS CONTROL
    # ------------------------------------------------------------
    all_vals = np.concatenate([
        df[df['StationID'].isin(common)][metric].dropna().values
        for df in dfs
        if metric in df.columns
    ])

    if len(all_vals) > 0:

        if cfg["clip"] is not None:
            ax.set_xlim(*cfg["clip"])

        elif metric_upper == "PBIAS":
            mx = np.max(np.abs(all_vals))
            ax.set_xlim(-mx * 1.05, mx * 1.05)

        else:
            mn, mx = np.min(all_vals), np.max(all_vals)
            pad = 0.02 * (mx - mn if mx > mn else 1)
            ax.set_xlim(mn - pad, mx + pad)

    # ------------------------------------------------------------
    # INSET
    # ------------------------------------------------------------
    if cfg["inset"]:

        valid = [s for s in outlier_series if len(s["x"]) > 0]

        if valid:
            inset = ax.inset_axes([0.06, 0.60, 0.32, 0.32])

            xmin = min(np.min(s["x"]) for s in valid)
            xmax = max(np.max(s["x"]) for s in valid)
            ymax = max(np.max(s["y"]) for s in valid)

            for s in valid:
                inset.scatter(
                    s["x"], s["y"],
                    s=18,
                    color=s["color"],
                    #edgecolors="k", # ✅ removed outline 
                    linewidths=0.4,
                    alpha=0.85
                )

            inset.set_xlim(xmin, xmax)
            inset.set_ylim(-0.02, ymax + 0.05)

            inset.set_title(
                f"{sum(len(s['x']) for s in valid)} outlier(s)",
                fontsize=7
            )

            inset.tick_params(labelsize=6)
            inset.grid(True, alpha=0.3)

    # ------------------------------------------------------------
    # STYLE
    # ------------------------------------------------------------
    ax.set_title(
        f"eCDF of {metric} ({len(common)} stations)",
        fontsize=16,
        fontweight="bold"
    )

    ax.set_xlabel(
        f"{metric} {_metric_unit(metric)}",
        fontsize=14,
        fontweight="bold"
    )

    ax.set_ylabel(
        "Cumulative probability",
        fontsize=14,
        fontweight="bold"
    )

    ax.set_ylim(0, 1.02)

    ax.grid(True, linestyle="--", alpha=0.7)

    ax.legend(
        title=legend_title,
        loc="lower right",
        fontsize=13,
        title_fontsize=14,
        frameon=False
    )

    fig.tight_layout()
    fig.savefig(output_file, dpi=150, bbox_inches="tight")
    plt.close(fig)

    print(f"Saved: {output_file.name}")

# =============================================================================
# SPATIAL PLOTTING (ADDED ✅)
# =============================================================================
def _normalize_basin_path(p: Path) -> Path:
    p = Path(p)
    return p if p.suffix.lower() == ".shp" else p.with_suffix(".shp")


def plot_spatial_stations_metrics(
    cache,
    coords: pd.DataFrame,
    metric='KGE',
    run_index=0,
    run_labels: Optional[List[str]] = None,
    output_file='spatial.png',
    common_stations: Optional[Set[str]] = None,
    basin_shapefile: Optional[Path] = None,
    point_size=10,
    basin_linewidth=0.4,
    alpha=0.8,
    dpi=300
):

    output_file = Path(output_file)

    if not cache.dfs:
        print("No runs available for spatial plot.")
        return

    run_index = min(max(run_index, 0), len(cache.dfs) - 1)

    # ------------------------------------------------------------------
    # Run label
    # ------------------------------------------------------------------
    if run_labels and run_index < len(run_labels):
        name = run_labels[run_index]
    else:
        name = cache.run_names[run_index]

    # ------------------------------------------------------------------
    # Metric dataframe
    # ------------------------------------------------------------------
    df = cache.dfs[run_index][['StationID', metric]].copy()
    df['StationID'] = df['StationID'].astype(str)

    coords = coords.copy()
    coords['StationID'] = coords['StationID'].astype(str)

    # ------------------------------------------------------------------
    # Common stations
    # ------------------------------------------------------------------
    _, common, _ = cache.get_metric_data(
        metric,
        allowed_stations=common_stations
    )

    if not common:
        print("No common stations for spatial plotting.")
        return

    plot_df = (
        coords[coords['StationID'].isin(common)]
        .merge(df, on='StationID')
        .dropna(subset=[metric])
    )

    if plot_df.empty:
        print("No spatial data available.")
        return

    # ------------------------------------------------------------------
    # Optional clipping
    # ------------------------------------------------------------------
    if metric in ["NKGE", "NKGE2012"]:
        plot_df[metric] = plot_df[metric].clip(0, 1)

    # ------------------------------------------------------------------
    # Styling
    # ------------------------------------------------------------------
    plt.rcParams.update({
        "font.size": 11,
        "axes.labelsize": 12,
        "axes.titlesize": 13,
        "xtick.labelsize": 10,
        "ytick.labelsize": 10,
        "axes.linewidth": 0.8,
    })

    # ------------------------------------------------------------------
    # Figure + GridSpec
    # ------------------------------------------------------------------
    fig = plt.figure(
        figsize=(8.5, 7.0),
        constrained_layout=False
    )

    gs = GridSpec(
        1,
        2,
        figure=fig,
        width_ratios=[1, 0.045],
        wspace=0.05
    )

    ax = fig.add_subplot(gs[0])
    cax = fig.add_subplot(gs[1])

    # ------------------------------------------------------------------
    # Basin overlay
    # ------------------------------------------------------------------
    if basin_shapefile:
        bp = _normalize_basin_path(basin_shapefile)

        if bp.exists():
            try:
                basin = gpd.read_file(bp)

                if basin.crs and basin.crs.to_string() != "EPSG:4326":
                    basin = basin.to_crs(4326)

                basin.boundary.plot(
                    ax=ax,
                    color='black',
                    linewidth=basin_linewidth,
                    zorder=5
                )

            except Exception as e:
                print(f"Warning: Basin load failed: {e}")

    # ------------------------------------------------------------------
    # Colormap setup
    # ------------------------------------------------------------------
    cmap = 'viridis'
    norm = None

    if metric == "PBIAS":

        vmax = max(
            abs(plot_df[metric].min()),
            abs(plot_df[metric].max())
        )

        norm = colors.TwoSlopeNorm(
            vmin=-vmax,
            vcenter=0,
            vmax=vmax
        )

        cmap = 'RdBu_r'

    # ------------------------------------------------------------------
    # Scatter plot
    # ------------------------------------------------------------------
    sc = ax.scatter(
        plot_df['Longitude'],
        plot_df['Latitude'],
        c=plot_df[metric],
        cmap=cmap,
        norm=norm,
        s=point_size,
        edgecolors='None',
        linewidths=0.2,
        alpha=alpha,
        zorder=2
    )

    # ------------------------------------------------------------------
    # Tight spatial extent
    # ------------------------------------------------------------------
    xmin = plot_df['Longitude'].min()
    xmax = plot_df['Longitude'].max()

    ymin = plot_df['Latitude'].min()
    ymax = plot_df['Latitude'].max()

    # Expand with basin bounds if available
    if basin_shapefile and 'basin' in locals():

        bxmin, bymin, bxmax, bymax = basin.total_bounds

        xmin = min(xmin, bxmin)
        xmax = max(xmax, bxmax)

        ymin = min(ymin, bymin)
        ymax = max(ymax, bymax)

    # Padding
    xpad = (xmax - xmin) * 0.03
    ypad = (ymax - ymin) * 0.03

    ax.set_xlim(xmin - xpad, xmax + xpad)
    ax.set_ylim(ymin - ypad, ymax + ypad)

    # Efficient use of figure space
    ax.set_aspect('auto')

    # ------------------------------------------------------------------
    # Colorbar
    # ------------------------------------------------------------------
    cbar = fig.colorbar(
        sc,
        cax=cax
    )

    cbar.set_label(
        metric,
        rotation=90,
        labelpad=10
    )

    cbar.outline.set_linewidth(0.6)

    # ------------------------------------------------------------------
    # Labels and title
    # ------------------------------------------------------------------
    ax.set_title(
        f"{metric} | {name} ({len(plot_df)} stations)",
        pad=10
    )

    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")

    # ------------------------------------------------------------------
    # Grid styling
    # ------------------------------------------------------------------
    ax.grid(
        True,
        linestyle='--',
        linewidth=0.4,
        alpha=0.4
    )

    for spine in ax.spines.values():
        spine.set_linewidth(0.8)

    # ------------------------------------------------------------------
    # Manual layout
    # Keeps colorbar aligned to plot box only
    # ------------------------------------------------------------------
    fig.subplots_adjust(
        left=0.10,
        right=0.88,
        bottom=0.11,
        top=0.92
    )

    # ------------------------------------------------------------------
    # Save
    # ------------------------------------------------------------------
    fig.savefig(
        output_file,
        dpi=dpi,
        facecolor='white'
    )

    plt.close(fig)

    print(f"Saved spatial plot: {output_file.name}")


    # ------------------------------------------------------------------
    # Spatial mapping
    # ------------------------------------------------------------------
def plot_spatial_station_metrics_comparison(
    cache,
    coords: pd.DataFrame,
    metric='KGE',
    run_index1=0,
    run_index2=1,
    run_labels: Optional[List[str]] = None,
    output_file='spatial_comparison.png',
    common_stations: Optional[Set[str]] = None,
    basin_shapefile: Optional[Path] = None,
    point_size=20,
    basin_linewidth=0.7,
    alpha=0.9,
    dpi=300
):

    output_file = Path(output_file)

    # ------------------------------------------------------------------
    # Validate runs
    # ------------------------------------------------------------------
    if len(cache.dfs) < 2:
        print("Need at least two runs for comparison.")
        return

    i1 = min(max(run_index1, 0), len(cache.dfs) - 1)
    i2 = min(max(run_index2, 0), len(cache.dfs) - 1)

    # ------------------------------------------------------------------
    # Labels
    # ------------------------------------------------------------------
    name1 = (
        run_labels[i1]
        if run_labels and i1 < len(run_labels)
        else cache.run_names[i1]
    )

    name2 = (
        run_labels[i2]
        if run_labels and i2 < len(run_labels)
        else cache.run_names[i2]
    )

    # ------------------------------------------------------------------
    # Metric data
    # ------------------------------------------------------------------
    df1 = (
        cache.dfs[i1][['StationID', metric]]
        .rename(columns={metric: "m1"})
    )

    df2 = (
        cache.dfs[i2][['StationID', metric]]
        .rename(columns={metric: "m2"})
    )

    df1['StationID'] = df1['StationID'].astype(str)
    df2['StationID'] = df2['StationID'].astype(str)

    coords = coords.copy()
    coords['StationID'] = coords['StationID'].astype(str)

    merged = df1.merge(df2, on='StationID')

    # ------------------------------------------------------------------
    # Common stations
    # ------------------------------------------------------------------
    _, common, _ = cache.get_metric_data(
        metric,
        allowed_stations=common_stations
    )

    if not common:
        print("No common stations for comparison.")
        return

    merged = merged[
        merged['StationID'].isin(common)
    ]

    # ------------------------------------------------------------------
    # Difference
    # ------------------------------------------------------------------
    merged['diff'] = merged['m1'] - merged['m2']

    plot_df = (
        coords.merge(merged, on='StationID')
        .dropna(subset=['diff'])
    )

    if plot_df.empty:
        print("No comparison data.")
        return

    # ------------------------------------------------------------------
    # Styling
    # ------------------------------------------------------------------
    plt.rcParams.update({
        "font.size": 11,
        "axes.labelsize": 12,
        "axes.titlesize": 13,
        "xtick.labelsize": 10,
        "ytick.labelsize": 10,
        "axes.linewidth": 0.8,
    })

    # ------------------------------------------------------------------
    # Figure + GridSpec
    # ------------------------------------------------------------------
    fig = plt.figure(
        figsize=(8.5, 7.0),
        constrained_layout=False
    )

    gs = GridSpec(
        1,
        2,
        figure=fig,
        width_ratios=[1, 0.045],
        wspace=0.05
    )

    ax = fig.add_subplot(gs[0])
    cax = fig.add_subplot(gs[1])

    # ------------------------------------------------------------------
    # Basin overlay
    # ------------------------------------------------------------------
    basin = None

    if basin_shapefile:

        bp = _normalize_basin_path(basin_shapefile)

        if bp.exists():

            try:
                basin = gpd.read_file(bp)

                if (
                    basin.crs and
                    basin.crs.to_string() != "EPSG:4326"
                ):
                    basin = basin.to_crs(4326)

            except Exception as e:
                print(f"Warning: Basin load failed: {e}")

    # ------------------------------------------------------------------
    # Symmetric color normalization
    # ------------------------------------------------------------------
    bound = max(
        abs(plot_df['diff'].min()),
        abs(plot_df['diff'].max()),
        0.1
    )

    norm = colors.TwoSlopeNorm(
        vmin=-bound,
        vcenter=0,
        vmax=bound
    )

    # ------------------------------------------------------------------
    # Scatter plot
    # ------------------------------------------------------------------
    sc = ax.scatter(
        plot_df['Longitude'],
        plot_df['Latitude'],
        c=plot_df['diff'],
        cmap='RdBu_r',
        norm=norm,
        s=point_size,
        edgecolors='None',
        linewidths=0.2,
        alpha=alpha,
        zorder=2
    )

    # ------------------------------------------------------------------
    # Plot basin LAST so boundaries stay visible
    # ------------------------------------------------------------------
    if basin is not None:

        basin.boundary.plot(
            ax=ax,
            color='black',
            linewidth=basin_linewidth,
            zorder=5
        )

    # ------------------------------------------------------------------
    # Spatial extent
    # Include both stations and basin
    # ------------------------------------------------------------------
    xmin = plot_df['Longitude'].min()
    xmax = plot_df['Longitude'].max()

    ymin = plot_df['Latitude'].min()
    ymax = plot_df['Latitude'].max()

    if basin is not None:

        bxmin, bymin, bxmax, bymax = basin.total_bounds

        xmin = min(xmin, bxmin)
        xmax = max(xmax, bxmax)

        ymin = min(ymin, bymin)
        ymax = max(ymax, bymax)

    xpad = (xmax - xmin) * 0.03
    ypad = (ymax - ymin) * 0.03

    ax.set_xlim(xmin - xpad, xmax + xpad)
    ax.set_ylim(ymin - ypad, ymax + ypad)

    # Efficient figure usage
    ax.set_aspect('auto')

    # ------------------------------------------------------------------
    # Colorbar
    # ------------------------------------------------------------------
    cbar = fig.colorbar(
        sc,
        cax=cax
    )

    cbar.set_label(
        f"{metric} difference",
        rotation=90,
        labelpad=10
    )

    cbar.outline.set_linewidth(0.6)

    # ------------------------------------------------------------------
    # Labels / title
    # ------------------------------------------------------------------
    ax.set_title(
        f"{metric}: {name1} − {name2}",
        pad=10
    )

    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")

    # ------------------------------------------------------------------
    # Grid styling
    # ------------------------------------------------------------------
    ax.grid(
        True,
        linestyle='--',
        linewidth=0.4,
        alpha=0.4
    )

    for spine in ax.spines.values():
        spine.set_linewidth(0.8)

    # ------------------------------------------------------------------
    # Manual layout
    # Keeps colorbar aligned with plot box
    # ------------------------------------------------------------------
    fig.subplots_adjust(
        left=0.10,
        right=0.88,
        bottom=0.11,
        top=0.92
    )

    # ------------------------------------------------------------------
    # Save
    # ------------------------------------------------------------------
    fig.savefig(
        output_file,
        dpi=dpi,
        facecolor='white'
    )

    plt.close(fig)

    print(f"Saved comparison: {output_file.name}")

In [8]:
# =============================================================================
# USAGE
# =============================================================================
if __name__ == "__main__":

    # ------------------------------------------------------------------
    # Paths & settings
    # ------------------------------------------------------------------
    base_path = Path(r"D:\Zelalem\RUNs")
    
    station_id_col = "Obs_NM"

    stations_file = Path(
        r"D:\Zelalem\WSC\combined_discharge_stations_comids.gpkg"
    )

    basin_file = Path(
        r"D:\Zelalem\WSC\CanTrans_MERIT_StudyDomain.shp"
    )

    date_column = "Date"

    start_date = "1980-10-01"

    end_date = "2024-09-30"

#    mesh_versions = [
#        "GSDE_MERIT",
#        "SoilGrid_MERIT"
#    ]
#    gru_types = [
#        "Lumped",
#        "Distributed"
#    ]

    mesh_versions = [
        "geofabric"
    ]
    gru_types = [
        "MERIT-CLRH"
    ]

    skip_days = 365

    missing_value = -1

    # ------------------------------------------------------------------
    # Labels
    # ------------------------------------------------------------------
    labels = [
        "MERIT-NALCMS-GSDE-CaSRv3.2",
        "CLRH-NALCMS-GSDE-CaSRv3.2"
    ]

    # ------------------------------------------------------------------
    # Load station coordinates
    # ------------------------------------------------------------------
    coords = load_stations_with_filters(
        stations_file,
        station_id_col=station_id_col,
        filters=[
            ("PRecord", ">= 10"),
#            ("Sub_Reg", "== 'QC'")
#            ("PRecord", ">= 10"),
#            ("DA_Diff", "> -10"),
#            ("DA_Diff", "< 10"),
            ("HYD_STATUS", "== 'A'")
        ]
    )

    # ------------------------------------------------------------------
    # Build metrics CSVs
    # ------------------------------------------------------------------
    print("Processing all model runs...")

    output_csvs = process_all_runs(
        base_path,
        mesh_versions,
        gru_types,
        skip_days=skip_days,
        missing_value=missing_value,
        date_column=date_column,
        start_date=start_date,
        end_date=end_date
    )

    if not output_csvs:
        raise RuntimeError("No output CSVs generated.")

    # ------------------------------------------------------------------
    # Load cache
    # ------------------------------------------------------------------
    print("\nLoading and filtering stations...")

    cache = MetricsCache(output_csvs)
    
    print("\n--- RUN ↔ LABEL MAPPING ---")
    for i, (name, label) in enumerate(zip(cache.run_names, labels)):
        print(f"{i}: {label}  ←  {name}")

    # ------------------------------------------------------------------
    # Common stations across runs
    # ------------------------------------------------------------------
    allowed = set(coords["StationID"].astype(str))

    metric_to_use = "KGE"

    common = cache.common_stations(
        metric=metric_to_use,
        allowed_stations=allowed
    )

    print(
        f"Using {len(common)} common stations "
        f"across runs for metric={metric_to_use}"
    )


    # ------------------------------------------------------------------
    # Output directory
    # ------------------------------------------------------------------
    figs = Path("./figs_geofabric")

    figs.mkdir(
        parents=True,
        exist_ok=True
    )


    # ------------------------------------------------------------------
    # Plot
    # ------------------------------------------------------------------
    plot_cumulative_distribution(
        cache,
        metric=metric_to_use,
        legend_title="Model Runs",
        legend_labels=labels,
        line_colors=[
            "black",
            "blue",
            "red",
            "green"
        ],
        line_width=1.25,        
        output_file=figs / f"cdf_{metric_to_use.lower()}.png",
        common_stations=common
    )


    # ------------------------------------------------------------------
    # Spatial Plot metrics and comparion
    # ------------------------------------------------------------------
    # The study domain boundary
    basin_file = Path(r"D:\Zelalem\WSC\CanTrans_MERIT_StudyDomain.shp")

    # Select the run to plot
    run_index = 0

    # Plot the spatial distribution of the performance metrics
    plot_spatial_stations_metrics(
        cache=cache,
        coords=coords,
        metric=metric_to_use,
        run_index=run_index,
        run_labels=labels,
        basin_shapefile=basin_file,
        output_file=figs / f"spatial_run{run_index}_{metric_to_use.lower()}.png",
        point_size=15
    )

    # Select how to compare the spatial distribution of the performance metrics [order of comparision]
    run_index1 = 0
    run_index2 = 1
    plot_spatial_station_metrics_comparison(
        cache,
        coords,
        metric=metric_to_use,
        run_index1=run_index1,
        run_index2=run_index2,
        run_labels=labels,
        basin_shapefile=basin_file,
        output_file=figs / f"spatial_{metric_to_use.lower()}_diff_{run_index1}_{run_index2}_runs.png",
        point_size=15
    )

Filter PRecord >= 10: 6192→4961
Filter HYD_STATUS == 'A': 4961→3297
Processing all model runs...
Processing: MESH_output_streamflow.csv
⚠ 394 stations had issues in metrics_geofabric_MERIT-CLRH_Iteration_2.12.csv
Processing: MESH_output_streamflow.csv
⚠ 2044 stations had issues in metrics_geofabric_MERIT-CLRH_Iteration_3.02.csv

Loading and filtering stations...
Loading 2 metric files...
Loaded 2 valid runs.

--- RUN ↔ LABEL MAPPING ---
0: MERIT-NALCMS-GSDE-CaSRv3.2  ←  Geofabric Merit-Clrh Iteration 2.12
1: CLRH-NALCMS-GSDE-CaSRv3.2  ←  Geofabric Merit-Clrh Iteration 3.02
Excluded 3884 stations → excluded_stations_kge.txt
Common 2308 stations → common_stations_kge.txt
Using 2308 common stations across runs for metric=KGE
Excluded 3884 stations → excluded_stations_kge.txt
Common 2308 stations → common_stations_kge.txt
Saved: cdf_kge.png


### For plotting from an existing / previously calculated metrics

In [105]:
    # Ploting Soildata

    from pathlib import Path

    metrics_dir = Path("D:/Zelalem/RUNs/Soildata")

    csv_files = sorted(metrics_dir.rglob("metrics_*.csv"))
    print(f"Found {len(csv_files)} metric files")
    
    cache = MetricsCache(csv_files)

    print("\n--- RUN ↔ LABEL MAPPING ---")
    for i, (name, label) in enumerate(zip(cache.run_names, labels)):
        print(f"{i}: {label}  ←  {name}")

    coords = load_stations_with_filters(
        stations_file,
        station_id_col="Obs_NM",
        filters=[]
    )

    coords["StationID"] = coords["StationID"].astype(str).str.strip()
    allowed = set(coords["StationID"])


    metric_to_use = "NKGE2012"

    common = cache.common_stations(
        metric=metric_to_use,
        allowed_stations=allowed
    )

    print(
        f"Using {len(common)} common stations "
        f"across runs for metric={metric_to_use}"
    )


    # ------------------------------------------------------------------
    # Output directory
    # ------------------------------------------------------------------
    figs = Path("./figs_soil")

    figs.mkdir(
        parents=True,
        exist_ok=True
    )

    # ------------------------------------------------------------------
    # Labels
    # ------------------------------------------------------------------
    labels = [
        "Lumped-GSDE-MERIT-CaSRv3.2",
        "Lumped-SoilGrid-MERIT-CaSRv3.2",
        "Dist-SoilGrid-MERIT-CaSRv3.2"
    ]

    # ------------------------------------------------------------------
    # Plot
    # ------------------------------------------------------------------
    plot_cumulative_distribution(
        cache,
        metric=metric_to_use,
        legend_title="Model Runs",
        legend_labels=labels,
        line_colors=[
            "black",
            "blue",
            "red",
            "green"
        ],
        line_width=1.25,        
        output_file=figs / f"cdf_{metric_to_use.lower()}.png",
        common_stations=common
    )

Found 3 metric files

--- RUN ↔ LABEL MAPPING ---
0: Lumped-GSDE-MERIT-CaSRv3.2  ←  metrics_GSDE_MERIT_Lumped_Iteration_2.12
1: Lumped-SoilGrid-MERIT-CaSRv3.2  ←  metrics_SoilGrid_MERIT_Lumped_Iteration_3.05
2: Dist-SoilGrid-MERIT-CaSRv3.2  ←  metrics_SoilGrid_MERIT_Distributed_Iteration_3.06
Using 5798 common stations across runs for metric=NKGE2012
Saved: cdf_nkge2012.png


In [131]:
    # Ploting Land Cover

    from pathlib import Path

    metrics_dir = Path("D:/Zelalem/RUNs/landcover")

    csv_files = sorted(metrics_dir.rglob("metrics_*.csv"))
    print(f"Found {len(csv_files)} metric files")
    
    cache = MetricsCache(csv_files)

    # ------------------------------------------------------------------
    # Labels
    # ------------------------------------------------------------------
    labels = [
        "NALCMS-GSDE-MERIT-CaSRv3.2",
        "ESA-GSDE-MERIT-CaSRv3.2"
    ]

    print("\n--- RUN ↔ LABEL MAPPING ---")
    for i, (name, label) in enumerate(zip(cache.run_names, labels)):
        print(f"{i}: {label}  ←  {name}")

    coords = load_stations_with_filters(
        stations_file,
        station_id_col="Obs_NM",
        filters=[]
    )

    coords["StationID"] = coords["StationID"].astype(str).str.strip()
    allowed = set(coords["StationID"])


    metric_to_use = "PBIAS"

    common = cache.common_stations(
        metric=metric_to_use,
        allowed_stations=allowed
    )

    print(
        f"Using {len(common)} common stations "
        f"across runs for metric={metric_to_use}"
    )


    # ------------------------------------------------------------------
    # Output directory
    # ------------------------------------------------------------------
    figs = Path("./figs_landcover")

    figs.mkdir(
        parents=True,
        exist_ok=True
    )

    # ------------------------------------------------------------------
    # Plot
    # ------------------------------------------------------------------
    plot_cumulative_distribution(
        cache,
        metric=metric_to_use,
        legend_title="Model Runs",
        legend_labels=labels,
        line_colors=[
            "black",
            "red",
            "blue",
            "green"
        ],
        line_width=1.25,        
        output_file=figs / f"cdf_{metric_to_use.lower()}.png",
        common_stations=common
    )

Found 2 metric files

--- RUN ↔ LABEL MAPPING ---
0: NALCMS-GSDE-MERIT-CaSRv3.2  ←  metrics_landcover_NALCMS-ESA_Iteration_2.12
1: ESA-GSDE-MERIT-CaSRv3.2  ←  metrics_landcover_NALCMS-ESA_Iteration_3.04
Using 5872 common stations across runs for metric=PBIAS
Saved: cdf_pbias.png


In [125]:
    # Ploting Geofabric

    from pathlib import Path

    metrics_dir = Path("D:/Zelalem/RUNs/geofabric")

    csv_files = sorted(metrics_dir.rglob("metrics_*.csv"))
    print(f"Found {len(csv_files)} metric files")
    
    cache = MetricsCache(csv_files)

    print("\n--- RUN ↔ LABEL MAPPING ---")
    for i, (name, label) in enumerate(zip(cache.run_names, labels)):
        print(f"{i}: {label}  ←  {name}")

    coords = load_stations_with_filters(
        stations_file,
        station_id_col="Obs_NM",
        filters=[]
    )

    coords["StationID"] = coords["StationID"].astype(str).str.strip()
    allowed = set(coords["StationID"])


    metric_to_use = "NKGE2012"

    common = cache.common_stations(
        metric=metric_to_use,
        allowed_stations=allowed
    )

    print(
        f"Using {len(common)} common stations "
        f"across runs for metric={metric_to_use}"
    )


    # ------------------------------------------------------------------
    # Output directory
    # ------------------------------------------------------------------
    figs = Path("./figs_geofabric")

    figs.mkdir(
        parents=True,
        exist_ok=True
    )

    # ------------------------------------------------------------------
    # Labels
    # ------------------------------------------------------------------
    labels = [
        "MERIT-NALCMS-GSDE-CaSRv3.2",
        "CLRH-NALCMS-GSDE-CaSRv3.2"
    ]

    # ------------------------------------------------------------------
    # Plot
    # ------------------------------------------------------------------
    plot_cumulative_distribution(
        cache,
        metric=metric_to_use,
        legend_title="Model Runs",
        legend_labels=labels,
        line_colors=[
            "black",
            "red",
            "blue",
            "green"
        ],
        line_width=1.25,        
        output_file=figs / f"cdf_{metric_to_use.lower()}.png",
        common_stations=common
    )

Found 2 metric files

--- RUN ↔ LABEL MAPPING ---
0: MERIT-NALCMS-GSDE-CaSRv3.2  ←  metrics_geofabric_MERIT-CLRH_Iteration_2.12
1: CLRH-NALCMS-GSDE-CaSRv3.2  ←  metrics_geofabric_MERIT-CLRH_Iteration_3.02
Using 4085 common stations across runs for metric=NKGE2012
Saved: cdf_nkge2012.png


In [109]:
    # Ploting Climate Forcing

    from pathlib import Path

    metrics_dir = Path("D:/Zelalem/RUNs/Forcingdata")

    csv_files = sorted(metrics_dir.rglob("metrics_*.csv"))
    print(f"Found {len(csv_files)} metric files")
    
    cache = MetricsCache(csv_files)

    print("\n--- RUN ↔ LABEL MAPPING ---")
    for i, (name, label) in enumerate(zip(cache.run_names, labels)):
        print(f"{i}: {label}  ←  {name}")

    coords = load_stations_with_filters(
        stations_file,
        station_id_col="Obs_NM",
        filters=[]
    )

    coords["StationID"] = coords["StationID"].astype(str).str.strip()
    allowed = set(coords["StationID"])


    metric_to_use = "KGE2012"

    common = cache.common_stations(
        metric=metric_to_use,
        allowed_stations=allowed
    )

    print(
        f"Using {len(common)} common stations "
        f"across runs for metric={metric_to_use}"
    )


    # ------------------------------------------------------------------
    # Output directory
    # ------------------------------------------------------------------
    figs = Path("./figs_forcing")

    figs.mkdir(
        parents=True,
        exist_ok=True
    )

    # ------------------------------------------------------------------
    # Labels
    # ------------------------------------------------------------------
    labels = [
        "MERIT-MESH-CaSRv2.1",
        "MERIT-MESH-CaSRv3.1",
        "MERIT-MESH-CaSRv3.2",
        "MERIT-MESH-ERA5L"
    ]

    # ------------------------------------------------------------------
    # Plot
    #  colors = ["steelblue", "teal", "darkorange", "seagreen", "crimson"]
    # ------------------------------------------------------------------
    plot_cumulative_distribution(
        cache,
        metric=metric_to_use,
        legend_title="Model Runs",
        legend_labels=labels,
        line_colors=[
            "black",
            "teal",
            "red",
            "blue"
        ],
        line_width=1.25,        
        output_file=figs / f"cdf_{metric_to_use.lower()}.png",
        common_stations=common
    )

Found 4 metric files

--- RUN ↔ LABEL MAPPING ---
0: MERIT-MESH-CaSRv2.1  ←  metrics_Forcingdata_Average_GRU_Params_Iteration_2.01
1: MERIT-MESH-CaSRv3.1  ←  metrics_Forcingdata_Average_GRU_Params_Iteration_2.04
2: MERIT-MESH-CaSRv3.2  ←  metrics_Forcingdata_Average_GRU_Params_Iteration_2.12
3: MERIT-MESH-ERA5L  ←  metrics_Forcingdata_Average_GRU_Params_Iteration_2.13
Using 5737 common stations across runs for metric=KGE2012
Saved: cdf_kge2012.png


In [172]:
# Spatial ploting:
basin_file = Path(r"D:\Zelalem\WSC\CanTrans_MERIT_StudyDomain.shp")
# Plot with the SAME stations across all figures
figs = Path("./figs"); figs.mkdir(parents=True, exist_ok=True)
# Select the run to plot
run_index = 0
# Plot the spatial distribution of the performance metrics
plot_spatial_stations_metrics(
    cache=cache,
    coords=coords,
    metric=metric_to_use,
    run_index=run_index,
    run_labels=labels,
    basin_shapefile=basin_file,
    output_file=figs / f"spatial_run{run_index}_{metric_to_use.lower()}.png",
    point_size=15
)

# Plot the comparision of the spatial distribution of the performance metrics
run_index1 = 0
run_index2 = 1
plot_spatial_station_metrics_comparison(
    cache,
    coords,
    metric=metric_to_use,
    run_index1=run_index1,
    run_index2=run_index2,
    run_labels=labels,
    basin_shapefile=basin_file,
    output_file=figs / f"spatial_{metric_to_use.lower()}_diff_{run_index1}_{run_index2}_runs.png",
    point_size=15
)

Saved spatial plot: spatial_run0_pbias.png
Saved comparison: spatial_pbias_diff_0_1_runs.png


In [ ]:
# To plot all stat
metrics_to_plot = [
    "KGE",
    "KGE2012",
    "NKGE",
    "NKGE2012",
    "NSE",
    "PBIAS",
]

# 5) add labels
labels = [
    "Lumped-GSDE-MERIT-CaSRv3.2",
    "Dist-GSDE-MERIT-CaSRv3.2",
    "Lumped-SoilGrid-MERIT-CaSRv3.2",
    "Dist-SoilGrid-MERIT-CaSRv3.2"
]

figs = Path("./figs_landcover")
figs.mkdir(parents=True, exist_ok=True)

for metric_to_use in metrics_to_plot:

    print(f"Processing: {metric_to_use}")

    # ---- station intersection for THIS metric ----
    common = cache.common_stations(
        metric=metric_to_use,
        allowed_stations=set(coords["StationID"].astype(str))
    )

    if len(common) == 0:
        print(f"Skipping {metric_to_use}: no common stations")
        continue

    # ---- prepare data ----
    prepared, outliers, common_used = prepare_cdf_data(
        cache,
        metric=metric_to_use,
        common_stations=common,
        legend_labels=labels
    )

    # ---- plot ----
    plot_cdf_from_prepared(
        prepared,
        outliers,
        common_used,
        metric_to_use,
        figs / f"cdf_{metric_to_use.lower()}.png",
        legend_title="Model Runs"
    )